In [1]:
# build_features_v3 — 构建 v3 特征矩阵
# Canonical identifiers and targets + ESM2(1280) + structural features(7)

import pickle, numpy as np, pandas as pd

BASE = "/mnt/volume6/czj/labLGN/LabLZ/"
SRC  = BASE + "data_preparation/cell2024_model_single_subst.csv"
OUT  = BASE + "xgboost_trial/features_v3.csv"

df = pd.read_csv(SRC)

# The existing ESM2 cache uses Gene||Variant. Keep this key only for cache lookup.
df["legacy_esm_key"] = df["Gene"].astype(str) + "||" + df["Variant"].astype(str)
# Canonical identifier used by all newly generated feature and evaluation tables.
df["KEY"] = (df["Gene"].astype(str).str.strip() + "||"
                    + df["Mutation_used"].astype(str).str.strip())

assert len(df) == 2179, f"Expected 2179 modelling rows, found {len(df)}"
assert df["Mutation_used"].notna().all(), "Mutation_used contains missing values"
assert df["KEY"].is_unique, "KEY must be unique"
assert df["Mislocalized"].isin([0, 1]).all(), "Invalid binary target values"
assert int(df["Mislocalized"].sum()) == 236, "Expected 236 positive variants"

# ===== v3 多分类标签 =====
# 0: 不重定位 (Mislocalized=0)
# 1: C1_no_reloc → 同区室但测试过
# 2: C2_aggregation → 聚集
# 3: C3_secretory → 分泌途径 (ER/Golgi/囊泡/质膜)
# 4: C4_nuclear → 核定位
# 5: C5_cytoplasmic → 细胞质/线粒体

df["reloc_v3"] = 0  # 默认不重定位
df.loc[(df["label_5class"] == "C1_no_reloc"),     "reloc_v3"] = 1
df.loc[(df["label_5class"] == "C2_aggregation"), "reloc_v3"] = 2
df.loc[(df["label_5class"] == "C3_secretory"),   "reloc_v3"] = 3
df.loc[(df["label_5class"] == "C4_nuclear"),      "reloc_v3"] = 4
df.loc[(df["label_5class"] == "C5_cytoplasmic"),  "reloc_v3"] = 5

print("v3 多分类标签分布:")
class_names = ["不重定位(C0)", "同区室/C1", "聚集/C2", "分泌途径/C3", "核定位/C4", "细胞质/C5"]
for i in range(6):
    n = (df["reloc_v3"] == i).sum()
    print(f"  Class {i} ({class_names[i]:15s}): {n}")
print(f"  正例总计 (C1-C5): {(df['reloc_v3'] > 0).sum()}, base_rate={(df['reloc_v3'] > 0).sum()/len(df):.4f}")
assert (df["reloc_v3"].gt(0).astype(int) == df["Mislocalized"].astype(int)).all(), (
    "Phenotype-derived classes do not cover every source-study positive. "
    "Regenerate the data_preparation label outputs before building features."
)

# ===== 加载 ESM2 嵌入 =====
esm = pickle.load(open(BASE + "data_preparation/phase4_esm2_local_delta.pkl", "rb"))

# ===== 构建特征矩阵 =====
esm_cols = [f"esm_{i}" for i in range(1280)]

rows = []
missing_esm = []
for _, r in df.iterrows():
    e = esm.get(r["legacy_esm_key"])
    if e is None:
        missing_esm.append(r["KEY"])
        continue

    row = {
        "KEY": r["KEY"],
        "Gene": r["Gene"],
        "Mutation_used": r["Mutation_used"],
        "source": r["source"],
        "Mislocalized": int(r["Mislocalized"]),
        "phenotype_class": r["label_5class"],
        "reloc_v3": int(r["reloc_v3"]),
        # 结构特征 (来自 AlphaFold WT 结构)
        "plddt": r.get("plddt", np.nan),
        "sasa": r.get("sasa", np.nan),
        "rsa": r.get("rsa", np.nan),
        "ss_helix": r.get("ss_helix", np.nan),
        "ss_strand": r.get("ss_strand", np.nan),
        "ss_coil": r.get("ss_coil", np.nan),
        "delta_hydrophobicity": r.get("delta_hydrophobicity", np.nan),
        "struct_status": 1 if r.get("struct_status") == "ok" else 0,
        # 待计算: 突变体结构变化
        "ddg": np.nan,         # ΔΔG (需FoldX/Rosetta等)
        "plddt_diff": np.nan,  # pLDDT(MT) - pLDDT(WT) (需AlphaFold on MT)
    }
    # ESM2 嵌入
    for i, v in enumerate(e):
        row[esm_cols[i]] = float(v)
    rows.append(row)

assert not missing_esm, f"Missing ESM2 embeddings for {len(missing_esm)} variants: {missing_esm[:5]}"
feat = pd.DataFrame(rows)
assert len(feat) == len(df)
assert feat["KEY"].is_unique
assert int(feat["Mislocalized"].sum()) == 236
feat.to_csv(OUT, index=False, float_format="%.6g")

print(f"\n特征矩阵: {feat.shape[0]} 行 × {feat.shape[1]} 列")
print("Feature groups: ESM2(1280) + structural(7) + placeholders(2)")
print(f"有ddg值: {feat['ddg'].notna().sum()}/{len(feat)} (当前全NaN, 待算)")
print(f"有plddt_diff值: {feat['plddt_diff'].notna().sum()}/{len(feat)} (当前全NaN, 待算)")
print(f"有结构特征: {feat['plddt'].notna().sum()}/{len(feat)}")
print(f"\n→ 已保存 {OUT}")

v3 多分类标签分布:
  Class 0 (不重定位(C0)       ): 1943
  Class 1 (同区室/C1         ): 13
  Class 2 (聚集/C2          ): 34
  Class 3 (分泌途径/C3        ): 121
  Class 4 (核定位/C4         ): 29
  Class 5 (细胞质/C5         ): 39
  正例总计 (C1-C5): 236, base_rate=0.1083

特征矩阵: 2179 行 × 1297 列
Feature groups: ESM2(1280) + structural(7) + placeholders(2)
有ddg值: 0/2179 (当前全NaN, 待算)
有plddt_diff值: 0/2179 (当前全NaN, 待算)
有结构特征: 2168/2179

→ 已保存 /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/features_v3.csv
